# PureScale 4.0: Autonomous Multiscale Vision & Edge AI Engine
- **Autonomous Signal Quality Diagnostics**: Discrete Wavelet MAD noise, optical blur score, dynamic entropy, and dark channel haze index.
- **Multiscale Local Laplacian Pyramids**: 4-octave frequency band decomposition with halo-free micro-texture (L1) and structural contour (L2) synthesis.
- **Dark Channel Prior Atmospheric Dehazing**: Removes haze, fog, and smoke with Fast Guided Filter boundary preservation.
- **Multi-Cue Semantic Region Guidance**: Soft masks for Sky, Foliage, Skin, Shadow, and Structure.
- **Three Processing Engines**: PureDSP (100% deterministic, ~2 s at 1080p (see BENCH.md)), Neural AI (Compact Real-ESRGAN x4v3, ~4.87 MB), and Hybrid (AI + Multiscale DSP).
- **Zero Hallucination / Zero GPU Needed in PureDSP**: Operates purely on standard CPU.


In [ ]:
# @title PureScale 4.0: Image Enhancement Parameters
# @markdown ---
# @markdown ### [0] Enhancement Engine Mode
# @markdown **processing_mode** (default: PureDSP):
# @markdown - **PureDSP**: 100% deterministic mathematical pipeline (~2 s at 1080p (see BENCH.md), zero neural weights, zero hallucination).
# @markdown - **Neural AI**: Compact ~4.87 MB Real-ESRGAN General x4v3 edge super-resolution.
# @markdown - **Hybrid**: Neural edge synthesis + Local Laplacian + BIMEF + Oklab color + CAS clarity.
processing_mode = "PureDSP" # @param ["PureDSP", "Neural AI", "Hybrid"]

# @markdown ---
# @markdown ### [0.2] Autonomous Diagnostics & Auto-Tuning
auto_tune = False # @param {type:"boolean"}

# @markdown ---
# @markdown ### [0.4] Atmospheric Dehazing (Dark Channel Prior)
enable_dehaze = False # @param {type:"boolean"}
dehaze_strength = 0.50 # @param {"type":"slider","min":0.0,"max":1.0,"step":0.05}

# @markdown ---
# @markdown ### [0.6] Multiscale Local Laplacian Pyramid Filtering
enable_pyramid = True # @param {type:"boolean"}
pyramid_micro_texture = 1.20 # @param {"type":"slider","min":0.5,"max":2.0,"step":0.05}
pyramid_structure_boost = 1.10 # @param {"type":"slider","min":0.8,"max":1.8,"step":0.05}

# @markdown ---
# @markdown ### [0.8] Pre-Restoration Conditioning (De-pixelate & Deblur)
depixel_strength = 0 # @param {"type":"slider","min":0,"max":100,"step":5}
deblur_strength = 0 # @param {"type":"slider","min":0,"max":100,"step":5}

# @markdown ---
# @markdown ### [1] Spatial Scaling
upscale_factor = 2.0 # @param [1.0, 1.5, 2.0, 3.0, 4.0] {type:"raw"}

# @markdown ---
# @markdown ### [2] Detail Clarity (CAS)
sharpen_strength = 1.1 # @param {"type":"slider","min":0.0,"max":3.0,"step":0.1}

# @markdown ---
# @markdown ### [3] Structural Denoising (SWF)
enable_denoise = True # @param {type:"boolean"}
denoise_intensity = 40 # @param {"type":"slider","min":10,"max":100,"step":5}

# @markdown ---
# @markdown ### [4] Dynamic Range Fusion (BIMEF) & Exposure
enable_contrast = True # @param {type:"boolean"}
contrast_boost = 1.8 # @param {"type":"slider","min":1.0,"max":4.0,"step":0.1}
brightness_shift = 0 # @param {"type":"slider","min":-50,"max":50,"step":5}

# @markdown ---
# @markdown ### [5] Perceptual Color & White Balance (Oklab & CAT16)
vibrance_boost = 1.10 # @param {"type":"slider","min":1.0,"max":1.5,"step":0.05}
color_temperature = 0 # @param {"type":"slider","min":-30,"max":30,"step":5}

# @markdown ---
# @markdown ### [6] Facial Retouching (YuNet + Fast Guided Filter)
portrait_smooth = 35 # @param {"type":"slider","min":0,"max":100,"step":5}
eye_clarity = 1.3 # @param {"type":"slider","min":1.0,"max":2.0,"step":0.1}

# -------------------------------------------------------------
# Pipeline Execution via purescale modular architecture
# -------------------------------------------------------------
import os
import sys
import cv2
import numpy as np
from PIL import Image
from IPython.display import display
try:
    from google.colab import files
    in_colab = True
except ImportError:
    in_colab = False

if "." not in sys.path:
    sys.path.insert(0, ".")

from purescale.config import PipelineConfig, ProcessingMode, DeviceTarget
from purescale.pipeline import PureScalePipeline
from purescale.cli import load_image_with_alpha, save_image_with_alpha

mode_map = {
    "PureDSP": ProcessingMode.PURE_DSP,
    "Neural AI": ProcessingMode.NEURAL_AI,
    "Hybrid": ProcessingMode.HYBRID,
}

cfg = PipelineConfig(
    mode=mode_map.get(processing_mode, ProcessingMode.PURE_DSP),
    device=DeviceTarget.AUTO,
    enable_diagnostics=True,
    auto_tune=auto_tune,
    enable_dehaze=enable_dehaze,
    dehaze_strength=float(dehaze_strength),
    enable_pyramid=enable_pyramid,
    pyramid_micro_texture=float(pyramid_micro_texture),
    pyramid_structure_boost=float(pyramid_structure_boost),
    enable_semantic_guidance=True,
    scale=float(upscale_factor),
    sharpen_strength=float(sharpen_strength),
    enable_denoise=enable_denoise,
    denoise_intensity=int(denoise_intensity),
    enable_contrast=enable_contrast,
    contrast_boost=float(contrast_boost),
    brightness_shift=int(brightness_shift),
    vibrance_boost=float(vibrance_boost),
    color_temperature=int(color_temperature),
    depixel_strength=int(depixel_strength),
    deblur_strength=int(deblur_strength),
    portrait_smooth=int(portrait_smooth),
    eye_clarity=float(eye_clarity),
    output_format="PNG",
)

pipeline = PureScalePipeline(config=cfg)

if in_colab:
    print("Please select an image file to enhance:")
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded.")
    else:
        filename = list(uploaded.keys())[0]
        bgr, alpha, exif, icc = load_image_with_alpha(filename, return_meta=True)
        orig_h, orig_w = bgr.shape[:2]

        def on_progress(stage, ratio):
            print(f"[{int(ratio * 100)}%] {stage}")

        res = pipeline.enhance(bgr, config=cfg, alpha=alpha, progress_callback=on_progress)
        if res.diagnostics:
            print(res.diagnostics.summary_table())

        base_name = os.path.splitext(filename)[0]
        out_name = f"{base_name}_enhanced.png"
        save_image_with_alpha(res.image, res.alpha, out_name, "PNG", exif=exif, icc_profile=icc)

        out_h, out_w = res.image.shape[:2]
        print(f"Completed in {res.latency_ms:.2f} ms ({res.backend_name})")
        print(f"Resolution: {orig_w}x{orig_h} -> {out_w}x{out_h} ({cfg.scale}x)")

        disp_rgb = cv2.cvtColor(res.image, cv2.COLOR_BGR2RGB)
        display(Image.fromarray(disp_rgb))
        files.download(out_name)
else:
    print("PureScale 4.0 Pipeline configured and ready for execution.")



In [ ]:
# @title Clear Storage
# @markdown Run this cell to delete uploaded and processed images from disk and free memory.

import os, glob, gc

deleted = 0
for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.bmp"):
    for f in glob.glob(ext):
        try:
            os.remove(f)
            deleted += 1
        except OSError:
            pass

gc.collect()
print(f"Removed {deleted} temporary file(s).")
